# SHAP explanations for AgroScan

SHAP assigns each image region a contribution to each class score (Shapley values). This notebook uses `shap.Explainer` with an **image masker**, which works with any model through the same `predict` function the API's preprocessing contract defines.

Compared with LIME, SHAP gives additive, per-class attributions; it is slower, so `MAX_EVALS` is kept small. Raise it for smoother maps.

Run top to bottom, or: `jupyter nbconvert --to notebook --execute notebooks/shap_example.ipynb`.

In [ ]:
# --- Parameters (edit these, nothing else needs changing) ---
IMAGE_DIR = "data/plantdoc_test"   # any folder of leaf images; read-only. Run `dvc pull` first if empty.
N_IMAGES = 4
SEED = 0                            # same seed -> same images -> repeatable
OUT_DIR = "reports/explainability"
MAX_EVALS = 500                     # more = smoother and slower
TOP_CLASSES = 2

In [ ]:
import os
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "models").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "notebooks"))
import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
from explain_utils import load_assets, display_image, make_predict_fn, pick_images  # noqa: E402
np.random.seed(SEED)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
model, meta, class_names = load_assets()
predict = make_predict_fn(model, meta)
paths = pick_images(IMAGE_DIR, N_IMAGES, SEED)
assert paths, f"No images in {IMAGE_DIR}: run `dvc pull` or change IMAGE_DIR"
print(f"model_version={meta.get('version')}  img_size={meta['img_size']}  images={len(paths)}")

In [ ]:
import shap

imgs = np.stack([display_image(p, meta) for p in paths]).astype(np.float32)
masker = shap.maskers.Image("inpaint_telea", imgs[0].shape)
explainer = shap.Explainer(predict, masker, output_names=class_names)
sv = explainer(imgs, max_evals=MAX_EVALS, batch_size=50, outputs=shap.Explanation.argsort.flip[:TOP_CLASSES])
shap.image_plot(sv, show=False)
plt.savefig(f"{OUT_DIR}/shap_examples.png", dpi=150, bbox_inches="tight")
plt.show()

## What to check
- Red = pushes the score **toward** the class, blue = **away**. Do the red regions sit on the diseased area?
- Compare with the LIME figure for the same seed: agreement is reassuring, disagreement is worth a sentence in the model card.
- Both figures are saved in `reports/explainability/`.